# Notebook 01 — Exploratory Data Analysis

Explore both datasets before any preprocessing.

In [ ]:
import sys; sys.path.insert(0, '..')
import pickle, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from collections import Counter
from sklearn.datasets import fetch_20newsgroups


## 1. 20 Newsgroups Dataset

In [ ]:
news = fetch_20newsgroups(subset='all', remove=('headers','footers','quotes'), random_state=42)
texts_ng, labels_ng, names_ng = news.data, news.target, news.target_names
print(f"Documents : {len(texts_ng):,}")
print(f"Categories: {len(names_ng)}")
print(f"\nCategory list:")
for i, n in enumerate(names_ng):
    cnt = int((np.array(labels_ng) == i).sum())
    print(f"  {i:2d}. {n:<35s} {cnt:4d} docs")


In [ ]:
# Document length distribution
lengths_ng = [len(t.split()) for t in texts_ng]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(lengths_ng, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Word Count'); axes[0].set_ylabel('Documents')
axes[0].set_title('20 Newsgroups — Document Length Distribution')
axes[0].axvline(np.median(lengths_ng), color='red', linestyle='--', label=f'Median {np.median(lengths_ng):.0f}')
axes[0].legend()
# Category sizes
cat_counts = Counter(labels_ng)
cats = [names_ng[i] for i in sorted(cat_counts)]
counts = [cat_counts[i] for i in sorted(cat_counts)]
axes[1].barh(cats, counts, color='steelblue')
axes[1].set_xlabel('Document Count')
axes[1].set_title('Documents per Category')
plt.tight_layout(); plt.savefig('../outputs/figures/eda_newsgroups.png', bbox_inches='tight'); plt.show()
print(f"Median length: {np.median(lengths_ng):.0f} words | Mean: {np.mean(lengths_ng):.0f} | Max: {max(lengths_ng):,}")


In [ ]:
# Top word frequencies (raw)
from collections import Counter
import re
all_words = re.findall(r'\b[a-z]{3,}\b', ' '.join(texts_ng).lower())
top50 = Counter(all_words).most_common(30)
words, freqs = zip(*top50)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(words, freqs, color='darkorange')
ax.set_xticklabels(words, rotation=60, ha='right')
ax.set_title('Top 30 Raw Words — 20 Newsgroups (before preprocessing)')
ax.set_ylabel('Frequency')
plt.tight_layout(); plt.savefig('../outputs/figures/eda_top_words_newsgroups.png', bbox_inches='tight'); plt.show()


## 2. Wikipedia People Dataset

In [ ]:
df_wiki = pd.read_csv('../data/raw/people_wiki.csv')
print(f"Articles  : {len(df_wiki):,}")
print(f"Columns   : {list(df_wiki.columns)}")
print(f"Null texts: {df_wiki['text'].isna().sum()}")
print("\nSample names:")
print(df_wiki['name'].head(10).tolist())


In [ ]:
lengths_wiki = df_wiki['text'].str.split().str.len().fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths_wiki, bins=40, color='seagreen', edgecolor='white')
ax.axvline(lengths_wiki.median(), color='red', linestyle='--', label=f'Median {lengths_wiki.median():.0f}')
ax.set_xlabel('Word Count'); ax.set_ylabel('Articles')
ax.set_title('Wikipedia People — Article Length Distribution')
ax.legend()
plt.tight_layout(); plt.savefig('../outputs/figures/eda_wikipedia.png', bbox_inches='tight'); plt.show()
print(f"Median: {lengths_wiki.median():.0f} | Mean: {lengths_wiki.mean():.0f} | Max: {lengths_wiki.max()}")


In [ ]:
# Show sample articles
for _, row in df_wiki.sample(3, random_state=42).iterrows():
    print(f"NAME: {row['name']}")
    print(f"TEXT: {row['text'][:300]}...")
    print()


## 3. Dataset Comparison Summary

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['20 Newsgroups', 'Wikipedia People'],
    'Documents': [len(texts_ng), len(df_wiki)],
    'Categories': [len(names_ng), 'N/A (biographical)'],
    'Median Doc Length': [np.median(lengths_ng), lengths_wiki.median()],
    'Mean Doc Length': [np.mean(lengths_ng), lengths_wiki.mean()],
})
print(summary.to_string(index=False))
